<a href="https://colab.research.google.com/github/AdisakJack/CMPE-259-Project-BALLPARK/blob/main/CMPE_259_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/jldbc/pybaseball.git
%cd pybaseball
!pip install -e .

Cloning into 'pybaseball'...
remote: Enumerating objects: 2842, done.
remote: Counting objects: 100% (873/873), done.
remote: Compressing objects: 100% (364/364), done.
remote: Total 2842 (delta 702), reused 509 (delta 509), pack-reused 1969 (from 3)
Receiving objects: 100% (2842/2842), 7.34 MiB | 12.04 MiB/s, done.
Resolving deltas: 100% (1733/1733), done.
/content/pybaseball
Obtaining file:///content/pybaseball
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.6/455.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.7 MB/s eta 0:00:00
  Building editable for pybaseball (pyproject.toml) ... done
  Created wheel for pybaseball: filename=pybaseball-2.2.7-0.editable-py3-none-any.whl size=8388 sha256=044b1755a7f9c15c474601bbe60e081862fa238163289dc1262893

In [ ]:
!pip install -q -U google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.5/320.5 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-datastore 2.26.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.5 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.30.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 whi

In [ ]:
import os
from google import genai
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

client = genai.Client()  # picks up the key from the environment

In [ ]:
import time
from google.genai import errors

def generate_with_retry(model, contents, config=None, max_tries=5):
    for attempt in range(max_tries):
        try:
            return client.models.generate_content(model=model, contents=contents, config=config)
        except (errors.ServerError, errors.ClientError) as e:
            code = getattr(e, "code", None)
            if code in (429, 503) and attempt < max_tries - 1:
                wait = 2 ** attempt * 5   # 5s, 10s, 20s, 40s
                print(f"{code}, retrying in {wait}s...")
                time.sleep(wait)
            else:
                raise

resp = generate_with_retry("gemini-3.8-flash", "Reply with the single word: connected")
print(resp.text)

connected


In [ ]:
from pybaseball import statcast

In [ ]:
statcast(start_dt="2026-06-24", end_dt="2026-06-25").columns

This is a large query, it may take a moment to complete


100%|██████████| 2/2 [00:01<00:00,  1.77it/s]


Index(['pitch_type', 'game_date', 'release_speed', 'release_pos_x',
       'release_pos_z', 'player_name', 'batter', 'pitcher', 'events',
       'description',
       ...
       'batter_days_until_next_game', 'api_break_z_with_gravity',
       'api_break_x_arm', 'api_break_x_batter_in', 'arm_angle', 'attack_angle',
       'attack_direction', 'swing_path_tilt',
       'intercept_ball_minus_batter_pos_x_inches',
       'intercept_ball_minus_batter_pos_y_inches'],
      dtype='object', length=119)

In [ ]:
from pybaseball import  playerid_lookup
from pybaseball import  statcast_pitcher


| Data | Function | Source |
|---|---|---|
| Pitch-level Statcast | `statcast`, `statcast_pitcher`, `statcast_batter` | Baseball Savant |
| Season pitching and batting stats | `pitching_stats`, `batting_stats` | FanGraphs |
| Season stats (alternative) | `pitching_stats_bref`, `batting_stats_bref` | Baseball Reference |
| Stats over a date range | `pitching_stats_range`, `batting_stats_range` | Baseball Reference |
| Team schedule and results | `schedule_and_record` | Baseball Reference |
| Standings | `standings` | Baseball Reference |



In [ ]:
playerid_lookup('ramos', 'heliot')

Gathering player lookup table. This may take a moment.


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,ramos,heliot,671218,ramoh002,ramoshe02,22515,2022.0,2026.0


In [ ]:
from pybaseball import playerid_lookup, batting_stats_bref, pitching_stats_bref
from pybaseball import cache

cache.enable()  # caches responses locally, which matters for BR rate limits

# 1. Resolve the player (handles name -> IDs, including the bbref ID)
ids = playerid_lookup("lee", "jung hoo")
print(ids[["name_last", "name_first", "key_mlbam", "key_bbref", "mlb_played_first", "mlb_played_last"]])

# 2. Pull every batter for a season from Baseball Reference
df_2026 = batting_stats_bref(2026)   # no argument = current season to date
df_2025 = batting_stats_bref(2025)

# 3. Filter to your player
lee_2026 = df_2026[df_2026["Name"].str.contains("Jung Hoo Lee", case=False)]
print(lee_2026[["Name", "Tm", "G", "PA", "BA", "OBP", "SLG", "OPS", "HR", "RBI"]])

Gathering player lookup table. This may take a moment.
  name_last name_first  key_mlbam key_bbref  mlb_played_first  mlb_played_last
0       lee   jung hoo     808982   leeju01            2024.0           2026.0
             Name             Tm    G   PA     BA   OBP    SLG    OPS  HR  RBI
332  Jung Hoo Lee  San Francisco  143  579  0.276  0.32  0.404  0.724  10   57


In [ ]:
from pybaseball import bwar_bat, playerid_lookup, cache

cache.enable()

war = bwar_bat()   # full table, all seasons. Large, so load it once and reuse it

# Resolve the player first so you can match on ID instead of name
ids = playerid_lookup("lee", "jung hoo")
mlbam = int(ids["key_mlbam"].iloc[0])

lee = war[war["mlb_ID"] == mlbam]
print(lee[["name_common", "year_ID", "team_ID", "PA", "WAR"]])

        name_common  year_ID team_ID   PA   WAR
64831  Jung Hoo Lee     2024     SFG  158  0.12
64832  Jung Hoo Lee     2025     SFG  617  1.70
64833  Jung Hoo Lee     2026     SFG  579  1.20


In [ ]:
import pybaseball
print([f for f in dir(pybaseball) if "bref" in f or "statcast" in f or "bwar" in f])
help(pybaseball.statcast_batter_expected_stats)  # shows parameters

['batting_stats_bref', 'bwar_bat', 'bwar_pitch', 'pitching_stats_bref', 'statcast', 'statcast_batter', 'statcast_batter_bat_tracking', 'statcast_batter_exitvelo_barrels', 'statcast_batter_expected_stats', 'statcast_batter_percentile_ranks', 'statcast_batter_pitch_arsenal', 'statcast_batter_run_value', 'statcast_catcher_framing', 'statcast_catcher_poptime', 'statcast_fielding', 'statcast_fielding_run_value', 'statcast_outfield_catch_prob', 'statcast_outfield_directional_oaa', 'statcast_outfielder_jump', 'statcast_outs_above_average', 'statcast_pitcher', 'statcast_pitcher_arsenal_stats', 'statcast_pitcher_exitvelo_barrels', 'statcast_pitcher_expected_stats', 'statcast_pitcher_percentile_ranks', 'statcast_pitcher_pitch_arsenal', 'statcast_pitcher_run_value', 'statcast_pitcher_spin_dir_comp', 'statcast_running', 'statcast_running_splits', 'statcast_single_game', 'statcast_sprint_speed', 'team_batting_bref', 'team_fielding_bref', 'team_pitching_bref']
Help on function statcast_batter_expect

**Baseball Reference**

| Function | What it gives you |
|---|---|
| `batting_stats_bref(season)` | All batters for a season |
| `pitching_stats_bref(season)` | All pitchers for a season |
| `batting_stats_range(start_dt, end_dt)` | Batters over a date range (useful for post-All-Star splits) |
| `pitching_stats_range(start_dt, end_dt)` | Pitchers over a date range |
| `bwar_bat()` / `bwar_pitch()` | bWAR datasets, all seasons |
| `team_batting_bref(team, start, end)` | Team batting by season |
| `team_pitching_bref(team, start, end)` | Team pitching by season |
| `team_fielding_bref(team, start, end)` | Team fielding by season |
| `schedule_and_record(season, team)` | A team's game log and results |
| `standings(season)` | Division standings |
| `get_splits(playerid, year, ...)` | Player splits (home/away, vs. lefties, etc.) using the bbref ID |
| `amateur_draft(year, round)` | Draft results |
| `top_prospects(team, player_type)` | Prospect lists |

**Baseball Savant (Statcast)**

| Function | What it gives you |
|---|---|
| `statcast(start_dt, end_dt)` | Pitch-by-pitch data for all players (large, so use short ranges) |
| `statcast_batter(start, end, player_id)` | Pitch-level data for one batter |
| `statcast_pitcher(start, end, player_id)` | Pitch-level data for one pitcher |
| `statcast_batter_expected_stats(year)` | xBA, xSLG, xwOBA (good for "who overperformed their metrics") |
| `statcast_pitcher_expected_stats(year)` | Same for pitchers |
| `statcast_batter_exitvelo_barrels(year)` | Exit velocity and barrel rates |
| `statcast_pitcher_exitvelo_barrels(year)` | Same for pitchers |
| `statcast_batter_percentile_ranks(year)` | Savant percentile rankings |
| `statcast_pitcher_percentile_ranks(year)` | Same for pitchers |
| `statcast_pitcher_pitch_arsenal(year)` | Pitch mix and results by pitch type |
| `statcast_batter_pitch_arsenal(year)` | How batters do against each pitch type (useful for your query 3) |
| `statcast_pitcher_pitch_movement(year)` | Pitch movement |
| `statcast_pitcher_active_spin(year)` | Active spin |
| `statcast_outs_above_average(year, pos)` | Defense (OAA) (useful for your query 20) |
| `statcast_outfield_catch_probability`, `statcast_outfield_directional_oaa`, `statcast_outfielder_jump` | More defensive metrics |
| `statcast_catcher_poptime(year)` | Catcher pop time |
| `statcast_sprint_speed(year)` | Sprint speed |
| `statcast_running_splits(year)` | Running splits |

